# 02_baseline

線形回帰とランダムフォレストでベースラインを作る．CV は月の中のブロック 4 分割で出し，比較用にランダム 5 分割でも出す．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import KFold, PredefinedSplit, cross_val_predict, cross_val_score

from bike_sharing.cv import get_folds
from bike_sharing.data import load_data, save_submission
from bike_sharing.visualize import set_style

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
set_style()

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## 特徴量と目的変数

In [ ]:
# datetime から年・月・時間・曜日を取り出す
# day は使わない（test の 20〜31 日は train に一度も出てこない）
for df in [train, test]:
    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["hour"] = df["datetime"].dt.hour
    df["dow"] = df["datetime"].dt.dayofweek

# casual と registered は test に無いので使えない
features = ["season", "holiday", "workingday", "weather", "temp", "atemp", "humidity", "windspeed",
            "year", "month", "hour", "dow"]
X = train[features]
X_test = test[features]

# RMSLE に合わせて log(1 + count) を学習する
y = np.log1p(train["count"])

print(X.shape, X_test.shape)
X.head()

## CV の分け方

In [ ]:
# 月の中のブロックで 4 分割（1〜5，6〜10，11〜14，15〜19 日）
block_cv = PredefinedSplit(get_folds(train))

# 比較用．1 時間ずつランダムに分ける
random_cv = KFold(n_splits=5, shuffle=True, random_state=0)

## CV スコア

log(1 + count) を学習しているので，RMSE がそのまま RMSLE になる．

In [ ]:
# 参考：何も学習しない（全部 log(1 + count) の平均で予測する）
dummy = DummyRegressor()
dummy_block = -cross_val_score(dummy, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
dummy_random = -cross_val_score(dummy, X, y, cv=random_cv, scoring="neg_root_mean_squared_error")
print("ブロック 4 分割:", dummy_block.round(4), " 平均", dummy_block.mean().round(4))
print("ランダム 5 分割:", dummy_random.round(4), " 平均", dummy_random.mean().round(4))

In [ ]:
lr = LinearRegression()
lr_block = -cross_val_score(lr, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
lr_random = -cross_val_score(lr, X, y, cv=random_cv, scoring="neg_root_mean_squared_error")
print("ブロック 4 分割:", lr_block.round(4), " 平均", lr_block.mean().round(4))
print("ランダム 5 分割:", lr_random.round(4), " 平均", lr_random.mean().round(4))

In [ ]:
# 設定は sklearn の既定のまま．結果が毎回同じになるように random_state だけ固定する
rf = RandomForestRegressor(random_state=0)
rf_block = -cross_val_score(rf, X, y, cv=block_cv, scoring="neg_root_mean_squared_error")
rf_random = -cross_val_score(rf, X, y, cv=random_cv, scoring="neg_root_mean_squared_error")
print("ブロック 4 分割:", rf_block.round(4), " 平均", rf_block.mean().round(4))
print("ランダム 5 分割:", rf_random.round(4), " 平均", rf_random.mean().round(4))

In [ ]:
result = pd.DataFrame({
    "ブロック 4 分割": [dummy_block.mean(), lr_block.mean(), rf_block.mean()],
    "ランダム 5 分割": [dummy_random.mean(), lr_random.mean(), rf_random.mean()],
}, index=["何も学習しない", "線形回帰", "ランダムフォレスト"])
result["差"] = result["ブロック 4 分割"] - result["ランダム 5 分割"]
result.round(4)

## CV の予測を図で見る

ブロック 4 分割で，各行が検証側になったときの予測（out-of-fold 予測）を使う．学習に使った行をそのまま予測すると，ランダムフォレストは実際よりよく見えてしまう．

In [ ]:
# 各行が検証側になったときの予測を集める
oof_lr = cross_val_predict(lr, X, y, cv=block_cv)
oof_rf = cross_val_predict(rf, X, y, cv=block_cv)

# 全部の行でまとめて出した RMSLE（上の CV は fold ごとの平均なので少しずれる）
print("線形回帰:", round(root_mean_squared_error(y, oof_lr), 4))
print("ランダムフォレスト:", round(root_mean_squared_error(y, oof_rf), 4))

# 台数に戻しておく
train["pred_lr"] = np.expm1(oof_lr)
train["pred_rf"] = np.expm1(oof_rf)

In [ ]:
# 実際と予測の散布図（log(1 + count) の目盛り）．点線に近いほど当たっている
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
ax[0].plot(y, oof_lr, ".", markersize=2, alpha=0.3)
ax[0].plot([0, 7], [0, 7], "--")
ax[0].set_title("線形回帰")
ax[0].set_xlabel("実際")
ax[0].set_ylabel("予測")
ax[1].plot(y, oof_rf, ".", markersize=2, alpha=0.3)
ax[1].plot([0, 7], [0, 7], "--")
ax[1].set_title("ランダムフォレスト")
ax[1].set_xlabel("実際")
plt.show()

In [ ]:
# 時間ごとの平均台数を，勤務日と土日祝に分けて比べる
wd = train[train["workingday"] == 1].groupby("hour")[["count", "pred_lr", "pred_rf"]].mean()
hd = train[train["workingday"] == 0].groupby("hour")[["count", "pred_lr", "pred_rf"]].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ax[0].plot(wd.index, wd["count"], marker=".", label="実際")
ax[0].plot(wd.index, wd["pred_lr"], marker=".", label="線形回帰")
ax[0].plot(wd.index, wd["pred_rf"], marker=".", label="ランダムフォレスト")
ax[1].plot(hd.index, hd["count"], marker=".", label="実際")
ax[1].plot(hd.index, hd["pred_lr"], marker=".", label="線形回帰")
ax[1].plot(hd.index, hd["pred_rf"], marker=".", label="ランダムフォレスト")
ax[0].set_title("勤務日")
ax[1].set_title("土日祝")
ax[0].set_xlabel("時間")
ax[1].set_xlabel("時間")
ax[0].set_ylabel("平均台数")
ax[0].legend()
plt.show()

In [ ]:
# 1 週間分（2012 年 6 月 4 日（月）〜10 日（日））を並べてみる
week = train[(train["datetime"] >= "2012-06-04") & (train["datetime"] < "2012-06-11")]

plt.figure(figsize=(14, 4))
plt.plot(week["datetime"], week["count"], label="実際")
plt.plot(week["datetime"], week["pred_lr"], label="線形回帰")
plt.plot(week["datetime"], week["pred_rf"], label="ランダムフォレスト")
plt.title("2012 年 6 月 4 日〜10 日")
plt.ylabel("台数")
plt.legend()
plt.show()

In [ ]:
# 時間ごとの RMSLE．どの時間で外しているか
err = pd.DataFrame({
    "hour": train["hour"],
    "線形回帰": (oof_lr - y) ** 2,
    "ランダムフォレスト": (oof_rf - y) ** 2,
})
err_hour = np.sqrt(err.groupby("hour").mean())

plt.figure(figsize=(8, 4))
plt.plot(err_hour.index, err_hour["線形回帰"], marker="o", label="線形回帰")
plt.plot(err_hour.index, err_hour["ランダムフォレスト"], marker="o", label="ランダムフォレスト")
plt.title("時間ごとの RMSLE")
plt.xlabel("時間")
plt.legend()
plt.show()

## 学習したモデルの中身

In [ ]:
# 全部の train で学習し直す
lr.fit(X, y)
rf.fit(X, y)

In [ ]:
# 線形回帰の係数（列ごとに単位が違うので，大きさはそのまま比べられない）
pd.Series(lr.coef_, index=features).round(3)

In [ ]:
importance = pd.Series(rf.feature_importances_, index=features).sort_values()

plt.figure(figsize=(6, 5))
plt.barh(importance.index, importance.values)
plt.title("ランダムフォレストの特徴量の重要度")
plt.show()

## 提出用のファイル

作っておくだけで，提出は Kaggle の画面からする．

In [ ]:
# log(1 + count) で予測しているので expm1 で戻す
pred_lr = np.expm1(lr.predict(X_test))
pred_rf = np.expm1(rf.predict(X_test))
print("線形回帰で 0 未満になった行:", (pred_lr < 0).sum())

# count は 0 以上でないといけないので，マイナスは 0 にする
pred_lr = pred_lr.clip(min=0)

save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_lr}), "02_baseline_lr")
save_submission(pd.DataFrame({"datetime": test["datetime"], "count": pred_rf}), "02_baseline_rf")

## test の予測を図で見る

In [ ]:
# 1 日ごとの平均台数．train は実際の値，test はランダムフォレストの予測
# 行が欠けている日があるので，合計ではなく平均にして，20 行未満の日は除く
test["pred_rf"] = pred_rf
tr = train.groupby(train["datetime"].dt.date)["count"].agg(["mean", "size"])
te = test.groupby(test["datetime"].dt.date)["pred_rf"].agg(["mean", "size"])
tr = tr[tr["size"] >= 20]
te = te[te["size"] >= 20]

plt.figure(figsize=(14, 4))
plt.plot(tr.index, tr["mean"], ".", label="train（実際）")
plt.plot(te.index, te["mean"], ".", label="test（ランダムフォレストの予測）")
plt.title("1 日ごとの平均台数")
plt.ylabel("1 時間あたりの台数")
plt.legend()
plt.show()